# FontDiffusion fd_cache generator (Colab GPU)

Generate handwritten-style chu-Nom character images for all 6 books in one batch.

**Requirement:** GPU runtime (Runtime → Change runtime type → T4 / A100). Diffusion inference is ~50× slower on CPU.

**Workflow:** Self-contained. Run cells top to bottom.

**Output:** 6 `fd_cache_<book>.zip` files (offered for browser download at the end). Unzip each into `prepared/<book>/fd_cache/` on your Mac, then run `./run_pipeline.sh --step 3 && --step 4`.

## 1. Verify GPU + install dependencies

In [ ]:
!nvidia-smi || echo '⚠️  NO GPU detected. Switch runtime: Runtime → Change runtime type → T4.'

%pip install -q \
    diffusers==0.38.0 accelerate einops kornia info-nce-pytorch \
    fonttools pygame scikit-image safetensors scipy \
    pyyaml pillow opencv-python-headless 2>&1 | tail -3

import torch
print(f'\nPyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## 2. Clone the repository

Public clone — no token needed. If the repo becomes private later, replace the URL with `https://<USERNAME>:<TOKEN>@github.com/truong571/GanNhanOCR.git`.

In [ ]:
import os, shutil, sys
from pathlib import Path

REPO_URL = 'https://github.com/truong571/GanNhanOCR.git'
PROJECT_ROOT = Path('/content/GanNhanOCR')

# `--recurse-submodules` is essential: font_diffusion/ is a git submodule
# (its own repo at https://github.com/dzungphieuluuky/FontDiffusion). Without
# this flag, Colab clones an empty font_diffusion/ directory and cell 5 fails
# with "Missing path: font_diffusion/fonts/NomNaTong-Regular.ttf".

def _is_submodule_populated(root: Path) -> bool:
    fd_fonts = root / 'font_diffusion' / 'fonts'
    if not fd_fonts.exists():
        return False
    return any(fd_fonts.glob('*.ttf'))


if PROJECT_ROOT.exists():
    print(f'Project already cloned. Pulling latest...')
    !cd {PROJECT_ROOT} && git pull --ff-only

    # Heal stale submodule state: if font_diffusion/ was created empty by an
    # earlier clone-without-recurse, `git submodule update --init` will refuse
    # to clone into a non-empty directory. Remove the empty shell first.
    fd_dir = PROJECT_ROOT / 'font_diffusion'
    if fd_dir.exists() and not _is_submodule_populated(PROJECT_ROOT):
        print(f'font_diffusion/ exists but submodule not populated — removing for fresh init')
        shutil.rmtree(fd_dir)

    !cd {PROJECT_ROOT} && git submodule update --init --recursive
else:
    !git clone --recurse-submodules --depth 1 {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'\nWorking dir: {os.getcwd()}')

# Sanity check submodule populated
n_fonts = sum(
    1 for _ in (PROJECT_ROOT / 'font_diffusion' / 'fonts').glob('*.ttf')
) if (PROJECT_ROOT / 'font_diffusion' / 'fonts').exists() else 0
n_fonts += sum(
    1 for _ in (PROJECT_ROOT / 'font_diffusion' / 'fonts').glob('*.otf')
) if (PROJECT_ROOT / 'font_diffusion' / 'fonts').exists() else 0
print(f'font_diffusion/fonts/ contains {n_fonts} font files')
assert n_fonts > 0, 'Submodule font_diffusion/ is empty — clone failed to recurse submodules.'

!ls -la

## 3. Verify FontDiffusion checkpoint files

Encoders are small (~40 MB total) and live in the cloned repo. The `unet.safetensors` is large (~300 MB) and is downloaded separately by cell 3a.

In [ ]:
CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'DRO-20260227-19P2' / 'checkpoint_step_6000'
FST_CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'FST' / 'checkpoint_step_1500'

REQUIRED = [
    'content_encoder.safetensors',
    'mss_encoder.safetensors',
    'fst_projection.safetensors',
    'original_style_projection.safetensors',
    'unet.safetensors',
]

missing = []
for fn in REQUIRED:
    p = CKPT_DIR / fn
    alt = FST_CKPT_DIR / fn
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f'  ✓ DRO/{fn:40} {size_mb:>7.1f} MB')
    elif alt.exists():
        size_mb = alt.stat().st_size / 1024 / 1024
        print(f'  ✓ FST/{fn:40} {size_mb:>7.1f} MB')
    else:
        print(f'  ✗ {fn:40}  MISSING (neither DRO nor FST)')
        missing.append(fn)

if missing:
    print(f'\n⚠️  {len(missing)} required file(s) missing. Run cell 3a to download.')

### 3a. Download missing `unet.safetensors` from HuggingFace Hub

**Background:** The DRO-20260227-19P2 Phase-2 training run was **local-only** and was NOT backed up to HF Hub. The original DRO `unet.safetensors` (~300 MB) is no longer recoverable.

**Workaround:** Use the FST Phase-1 `unet.safetensors` from `dzungpham/font-diffusion-weights` as a substitute. FST is the Phase-1 base — slightly less manuscript-specific than the lost DRO Phase-2, but produces serviceable handwritten-style images. The DRO encoder files (already in cloned repo) are still used for content/style conditioning.

Run this cell once. It also pre-stages other FST encoders so the loader has a complete fallback set.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

HF_REPO = 'dzungpham/font-diffusion-weights'

FST_CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Download FST unet (Phase 1 base) — single source of truth for unet weights
fst_unet = FST_CKPT_DIR / 'unet.safetensors'
if not fst_unet.exists():
    print(f'Downloading FST/checkpoint_step_1500/unet.safetensors from {HF_REPO} ...')
    cached = hf_hub_download(
        repo_id=HF_REPO,
        filename='FST/checkpoint_step_1500/unet.safetensors',
    )
    shutil.copy2(cached, fst_unet)
    size_mb = fst_unet.stat().st_size / 1024 / 1024
    print(f'  ✓ FST unet → {fst_unet} ({size_mb:.1f} MB)')
else:
    print(f'FST unet already present.')

# 2. Mirror FST unet under DRO ckpt dir (substitute for the lost DRO Phase 2 unet)
dro_unet = CKPT_DIR / 'unet.safetensors'
if not dro_unet.exists():
    shutil.copy2(fst_unet, dro_unet)
    print(f'  ✓ Copied FST unet → {dro_unet} (DRO substitute)')
else:
    print(f'DRO unet already present.')

# 3. Best-effort pull of remaining FST encoders (in case the loader prefers FST)
fst_extra = [
    'content_encoder.safetensors',
    'mss_encoder.safetensors',
    'fst_module.safetensors',
    'fst_projection.safetensors',
    'original_style_projection.safetensors',
    'style_encoder.safetensors',
]
for fn in fst_extra:
    dst = FST_CKPT_DIR / fn
    if dst.exists():
        continue
    try:
        cached = hf_hub_download(
            repo_id=HF_REPO,
            filename=f'FST/checkpoint_step_1500/{fn}',
        )
        shutil.copy2(cached, dst)
        print(f'  ✓ FST {fn}')
    except Exception as e:
        print(f'  · FST {fn}: skip ({type(e).__name__})')

print('\nCheckpoint setup complete.')

## 4. Refresh `colab_diffusion/exports/`

Re-runs `aggregate_chars` to make sure `chars_<book>.txt` and style images are in sync with the latest `prepared/` data. Idempotent.

In [ ]:
import json
EXPORTS = PROJECT_ROOT / 'colab_diffusion' / 'exports'

!python -m colab_diffusion.aggregate_chars

manifest_path = EXPORTS / 'MANIFEST.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing {manifest_path}. aggregate_chars failed.')

manifest = json.load(open(manifest_path))
print('\nBooks to generate:')
for book, info in manifest['books'].items():
    chars_count = info.get('chars_count', 0)
    has_style = info.get('style_image') is not None
    mark = '✓' if (chars_count > 0 and has_style) else '✗'
    print(f'  [{mark}] {book:25} {chars_count:>5} chars   style={"OK" if has_style else "MISSING"}')
print(f'\nTotal union: {manifest["union_chars"]} unique chars')

## 5. Load FontDiffusion (one-time, reused for all books)

Loading takes ~30-60 s on T4. Watch for `FileNotFoundError` — that means a checkpoint file is still missing. If you see it, re-run cell 3a.

In [ ]:
from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

ckpt_dir = manifest['fontdiffusion_ckpt_subpath']
phase1_dir = manifest['fontdiffusion_phase1_ckpt_subpath']
font_path = manifest['font_path_subpath']

for p in (ckpt_dir, phase1_dir, font_path):
    assert Path(p).exists(), f'Missing path: {p}'

print('Loading FontDiffusion (~1 minute on T4)...\n')
generator = FontDiffusionGenerator(
    ckpt_dir=ckpt_dir,
    phase1_ckpt_dir=phase1_dir,
    font_path=font_path,
    cache_dir='/tmp/fd_global',
)
print('\n✓ FontDiffusion ready.')

## 6. Generate fd_cache for each book

Generates one image per (char × book) since each book has its own style reference. Total ~10,500 images across 6 books on T4: ~30-50 minutes.

In [ ]:
import shutil, time, zipfile

OUT_ZIP_DIR = PROJECT_ROOT / 'colab_diffusion' / 'fd_cache_zips'
OUT_ZIP_DIR.mkdir(parents=True, exist_ok=True)

for book, info in manifest['books'].items():
    if not info.get('chars_file') or not info.get('style_image'):
        print(f'  [skip] {book}: missing chars or style image')
        continue

    chars_file = EXPORTS / info['chars_file']
    style_image = str(EXPORTS / info['style_image'])
    chars = [line.strip() for line in open(chars_file, encoding='utf-8') if line.strip()]

    cache_dir = PROJECT_ROOT / 'colab_diffusion' / f'_tmp_{book}'
    cache_dir.mkdir(parents=True, exist_ok=True)
    generator.cache_dir = str(cache_dir)

    t0 = time.time()
    print(f'\n>>> {book}: generating {len(chars)} chars (cache → {cache_dir.name}/)')
    generated = generator.generate(chars, style_image, style_name=book)
    elapsed = time.time() - t0
    rate = len(generated) / max(elapsed, 1)
    print(f'    Generated: {len(generated)}/{len(chars)}  in {elapsed:.0f}s  ({rate:.1f} chars/sec)')

    zip_path = OUT_ZIP_DIR / f'fd_cache_{book}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for png in cache_dir.rglob('U+*.png'):
            z.write(png, arcname=png.name)
    size_mb = zip_path.stat().st_size / 1024 / 1024
    print(f'    Zipped → {zip_path.name}  ({size_mb:.1f} MB)')

print('\n✓ All books done.')

## 7. Download zips to your local Mac

Each `files.download` opens a browser save dialog. After all 6 land in your Downloads folder, run on Mac:

```sh
cd /path/to/GanNhanOCR
for book in CacThanhTruyen2 CacThanhTruyen4 CacThanhTruyen11 \
            SachThanhTruyen2 SachThanhTruyen4 SachThanhTruyen11; do
  mkdir -p "prepared/$book/fd_cache"
  unzip -o ~/Downloads/"fd_cache_${book}.zip" -d "prepared/$book/fd_cache/"
done

./run_pipeline.sh --step 3 && ./run_pipeline.sh --step 4
```

In [ ]:
from google.colab import files
for z in sorted(OUT_ZIP_DIR.glob('fd_cache_*.zip')):
    size_mb = z.stat().st_size / 1024 / 1024
    print(f'Downloading {z.name} ({size_mb:.1f} MB)...')
    files.download(str(z))

### Alternative: copy zips to Google Drive (no browser save dialogs)

If your Colab session might disconnect or you want offline access:

In [ ]:
# Uncomment to use:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_OUT = Path('/content/drive/MyDrive/GanNhanOCR_fd_cache')
# DRIVE_OUT.mkdir(parents=True, exist_ok=True)
# for z in sorted(OUT_ZIP_DIR.glob('fd_cache_*.zip')):
#     shutil.copy2(z, DRIVE_OUT / z.name)
#     print(f'  → {DRIVE_OUT / z.name}')